# Round 1 - Technical 1 (SQL | Pyspark | Data Engineering Basics)

1. **What is Incremental load and how can you implement this?**
2. **What is the difference between a job and a task?**
3. **You have 200 GB data and want to load it into a table. What is the best way to do this?**




1. **What is Incremental Load and How Can You Implement This?**

Incremental load refers to loading **only new or updated data** since the last load, rather than reloading the entire dataset.  
This approach improves efficiency and reduces processing time.

**Implementation Example:**  
Suppose you have a `source_df` DataFrame (source data) and a `target_table` (destination table).  
You want to load only records with a timestamp greater than the last processed timestamp.

python
# Get the last processed timestamp from the target table
last_processed_timestamp = target_table_df.select(max('updated_at')).collect()[0][0]

# Filter source data for new or updated records
incremental_df = source_df.filter(col('updated_at') > last_processed_timestamp)

# Write incremental data to the target table
incremental_df.write.format('delta').mode('append').saveAsTable('target_table')

**Summary:**  
- Identify the last processed timestamp.
- Filter source data for records newer than this timestamp.
- Append only these records to the target table.

2. **What is the difference between a job and a task?**

- **Job:**  
  A job is a high-level action in Spark (such as `count`, `save`, or `collect`) triggered by the user.  
  Each job is divided into one or more stages.

- **Task:**  
  A task is the smallest unit of execution in Spark.  
  Each task represents a computation on a partition of data within a stage.  
  Multiple tasks run in parallel to complete a stage.

**Summary:**  
- A job consists of multiple stages.  
- Each stage consists of multiple tasks.  
- Tasks are executed in parallel to complete the job.

3. **You have 200 GB data and want to load it into a table. What is the best way to do this?**

- Use distributed processing (e.g., Spark) to parallelize the load.
- Write data in optimized formats like Parquet or Delta.
- Use partitioning to improve performance.
- Tune Spark configurations (e.g., `spark.sql.shuffle.partitions`, executor memory/cores) for large data loads.
- Use batch loading instead of single large writes to avoid driver memory issues.
- Validate data schema before loading to prevent schema mismatch errors.
- Monitor job progress and handle failures with retries or checkpoints.
- Example (PySpark):

python
# Repartition data for optimal parallelism
df = df.repartition(100)

# Write data to Delta table with schema overwrite and partitioning
df.write.format('delta') \
  .mode('overwrite') \
  .option('overwriteSchema', 'true') \
  .partitionBy('date_column') \
  .saveAsTable('target_table')

4. **Write a query to fix matches between each team and every other team.**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
data = ['KKR','RCB','MI','CSK']
schema = ['team']

df = spark.createDataFrame(data=data,schema=schema)
df.show()


In [0]:
final_df = df.alias('team1').join(df.alias('team2'),(col('team1.team') != col('team2.team')) & (col('team1.team') < col('team2.team')))
final_df.display()

In [0]:
# Output: 
+-----+-----+
|team1|team2|
+-----+-----+
|KKR  |RCB  |
|KKR  |MI   |
|MI   |RCB  |
|CSK  |KKR  |
|CSK  |RCB  |
|CSK  |MI   |
+-----+-----+
